# Bank Customer Churn: Exploratory Data Analysis
**KUBS Machine Learning Project: Step 2**

Goal: Predict whether a bank customer will churn (`Exited = 1`) or stay (`Exited = 0`).

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import class_weight

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42

## 2. Load & Parse Data

In [ ]:
def load_data(path: str) -> pd.DataFrame:
    """Load the churn CSV and drop columns that carry no predictive signal."""
    df = pd.read_csv(path)
    # RowNumber, CustomerId, and Surname are identifiers — not features
    df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
    return df

df = load_data('Churn_Modelling.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

## 3. Missing Values & Data Quality

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'None — dataset is complete.')

print('\nDuplicate rows:', df.duplicated().sum())

## 4. Target Variable — Class Balance

In [ ]:
churn_counts = df['Exited'].value_counts()
churn_pct    = df['Exited'].value_counts(normalize=True) * 100

print('Class distribution:')
print(pd.DataFrame({'Count': churn_counts, 'Percent (%)': churn_pct.round(1)}))

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Stayed (0)', 'Churned (1)'], churn_counts.values,
       color=['steelblue', 'tomato'], edgecolor='white', linewidth=1.2)
for i, v in enumerate(churn_counts.values):
    ax.text(i, v + 50, f'{v}\n({churn_pct.values[i]:.1f}%)', ha='center', fontsize=10)
ax.set_title('Target Class Distribution', fontsize=13)
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('fig_class_balance.png')
plt.show()

## 5. Descriptive Statistics

In [ ]:
df.describe().round(2)

## 6. Univariate Analysis — Numerical Features

In [ ]:
num_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        axes[i].hist(df[df['Exited'] == label][col], bins=30, alpha=0.6,
                     color=color, label=('Stayed' if label == 0 else 'Churned'),
                     edgecolor='white')
    axes[i].set_title(col, fontsize=11)
    axes[i].legend(fontsize=8)

fig.suptitle('Numerical Features — Distribution by Churn Status', fontsize=13)
plt.tight_layout()
plt.savefig('fig_num_distributions.png')
plt.show()

## 7. Categorical Features — Churn Rate

In [ ]:
cat_cols = ['Geography', 'Gender', 'HasCrCard', 'IsActiveMember']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, col in zip(axes, cat_cols):
    churn_rate = df.groupby(col)['Exited'].mean() * 100
    counts     = df.groupby(col)['Exited'].count()
    bars = ax.bar(churn_rate.index.astype(str), churn_rate.values,
                  color='tomato', edgecolor='white', alpha=0.85)
    for bar, (cat, rate) in zip(bars, churn_rate.items()):
        n = counts[cat]
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.5,
                f'{rate:.1f}%\n(n={n})', ha='center', fontsize=8)
    ax.set_title(f'Churn rate by {col}', fontsize=10)
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, churn_rate.max() * 1.35)

plt.suptitle('Categorical Features — Churn Rate', fontsize=13)
plt.tight_layout()
plt.savefig('fig_cat_churn_rates.png')
plt.show()

## 8. Correlation Heatmap

In [ ]:
# Encode categoricals temporarily for correlation
df_enc = df.copy()
df_enc['Geography'] = LabelEncoder().fit_transform(df_enc['Geography'])
df_enc['Gender']    = LabelEncoder().fit_transform(df_enc['Gender'])

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_enc.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix (all features + target)', fontsize=13)
plt.tight_layout()
plt.savefig('fig_correlation.png')
plt.show()

print('\nCorrelation with Exited (sorted):')
print(corr['Exited'].drop('Exited').sort_values(key=abs, ascending=False).round(3))

## 9. Featurization & Preprocessing

In [ ]:
def build_features(df: pd.DataFrame):
    """
    Build X (feature matrix) and y (label array) from the raw dataframe.

    Transformations applied:
    - One-hot encode Geography and Gender (drop_first to avoid multicollinearity)
    - All remaining numeric features passed through as-is

    Returns
    -------
    X : pd.DataFrame  shape (n_samples, n_features)
    y : np.ndarray    shape (n_samples,)  — binary labels (0 = stayed, 1 = churned)
    feature_names : list[str]
    """
    df = df.copy()

    # One-hot encode categorical columns
    df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True)

    y = df['Exited'].values
    X = df.drop(columns=['Exited'])

    return X, y, list(X.columns)


X, y, feature_names = build_features(df)
print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'Features: {feature_names}')

## 10. Train / Validation / Test Split (80 / 10 / 10)

In [ ]:
def split_data(X, y, val_size=0.10, test_size=0.10, random_state=RANDOM_STATE):
    """
    Stratified split into train / val / test.
    Stratification preserves the churn class ratio in each split.
    """
    # First cut off the test set
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    # Then cut the validation set from the remaining data
    val_frac = val_size / (1 - test_size)   # relative proportion in trainval
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=val_frac, stratify=y_trainval,
        random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

for name, ys in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    churn_pct = ys.mean() * 100
    print(f'{name:5s}: {len(ys):5d} samples  |  churn rate = {churn_pct:.1f}%')

## 11. Feature Scaling

In [ ]:
# Scale only on train, then apply to val and test to avoid data leakage
scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print('Training set scaled. Means ≈ 0, Stds ≈ 1:')
print(f'  mean: {X_train_sc.mean(axis=0).round(4)}')
print(f'  std : {X_train_sc.std(axis=0).round(4)}')

## 12. Dataset Summary Statistics

In [ ]:
print('=' * 55)
print('DATASET SUMMARY')
print('=' * 55)
print(f'Total samples          : {len(df):,}')
print(f'Features (after OHE)   : {X.shape[1]}')
print(f'Target                 : Exited  (0 = stayed, 1 = churned)')
print(f'Class imbalance        : {y.mean()*100:.1f}% churn rate')
print()
print(f'Train set              : {len(y_train):,} samples ({len(y_train)/len(y)*100:.0f}%)')
print(f'Validation set         : {len(y_val):,} samples ({len(y_val)/len(y)*100:.0f}%)')
print(f'Test set               : {len(y_test):,} samples ({len(y_test)/len(y)*100:.0f}%)')
print()
print('Key findings from EDA:')
print('  • Age is the strongest numeric predictor (older → higher churn)')
print('  • IsActiveMember: inactive members churn ~2× more')
print('  • Germany has notably higher churn rate than France/Spain')
print('  • Female customers churn slightly more than male')
print('  • Balance = 0 is common among stayers; churners hold more balanced accounts')
print('  • NumOfProducts > 2 strongly predicts churn')
print('  • No missing values; no duplicates')
print('  • Dataset is imbalanced (~20% churn) — stratified splits used')

## 13. Export Processed Data (Optional)

In [ ]:
import os
os.makedirs('data_splits', exist_ok=True)

np.save('data_splits/X_train.npy', X_train_sc)
np.save('data_splits/X_val.npy',   X_val_sc)
np.save('data_splits/X_test.npy',  X_test_sc)
np.save('data_splits/y_train.npy', y_train)
np.save('data_splits/y_val.npy',   y_val)
np.save('data_splits/y_test.npy',  y_test)

print('Saved scaled splits to data_splits/')